In [ ]:
import os
import re
import csv
import json
import time
from groq import Groq
from datasets import load_dataset
from openai import OpenAI

In [ ]:
API_KEY = "Your key here"

router_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
groq_client = groq_client = Groq(api_key=API_KEY)

In [ ]:
MODELS = [
    ("groq", "llama-3.1-8b-instant"),
    ("groq", "llama-3.3-70b-versatile"),
    ("groq", "openai/gpt-oss-20b"),
    ("groq", "openai/gpt-oss-120b"),
    ("router","poolside/laguna-s-2.1:free"),
    ("router","google/gemma-4-26b-a4b-it:free"), 
    ("router","nvidia/nemotron-3-super-120b-a12b:free"),
    ("router","microsoft/phi-4"),
    ("router","qwen/qwen3-8b"),
    ("router","mistralai/ministral-8b-2512")

]

N_QUESTIONS = 1000
LETTERS = ["A", "B", "C", "D"]
REASONING_MODELS = ("gpt-oss", "deepseek", "laguna", "nemotron", "gemma")
HYBRID_MODELS = ("qwen3", "qwen2.5")

CSV_PATH = "data/mmlu_benchmark_results.csv"
QUESTIONS_JSON_PATH = "data/selected_questions.json"

In [ ]:
if os.path.exists(QUESTIONS_JSON_PATH):
    print(f"Loading cached dataset from {QUESTIONS_JSON_PATH}...")
    with open(QUESTIONS_JSON_PATH, "r", encoding="utf-8") as f:
        saved_items = json.load(f)
else:
    saved_items = []


if len(saved_items) < N_QUESTIONS:
    print(f"Expanding dataset cache from {len(saved_items)} to {N_QUESTIONS} questions...")
    # Load MMLU test split
    raw_dataset = load_dataset("cais/mmlu", "all", split="test")
    full_shuffled = raw_dataset.shuffle(seed=42)
    
    for i in range(len(saved_items), N_QUESTIONS):
        item = full_shuffled[i]
        saved_items.append({
            "question_id": f"Q{i+1}",
            "question": item["question"],
            "choices": item["choices"],
            "answer": LETTERS[item["answer"]]
        })
        
    with open(QUESTIONS_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(saved_items, f, indent=2)



existing_data = {item["question_id"]: {} for item in saved_items}

# Track all models (from the current list)
all_known_models = set(m[1] for m in MODELS)

if os.path.exists(CSV_PATH):
    print(f"Reading existing results from {CSV_PATH}...")
    with open(CSV_PATH, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames:
            # Memorize any old models in the CSV, even if they aren't in MODELS right now
            for col in reader.fieldnames:
                if col not in ["question_id", "correct_label"]:
                    all_known_models.add(col)
                    
            for row in reader:
                q_id = row["question_id"]
                # Safety catch just in case there's an orphaned row in the CSV
                if q_id not in existing_data:
                    existing_data[q_id] = {} 
                    
                for model in all_known_models:
                    if model in row and row[model] != "":
                        existing_data[q_id][model] = row[model]


In [ ]:
def format_question(item):
    choices = "\n".join(f"{LETTERS[idx]}. {c}" for idx, c in enumerate(item["choices"]))
    return (
        f"Answer the following multiple-choice question. "
        f"Reply with ONLY the letter (A, B, C, or D) of the correct answer and nothing else, do not explain or repeat the question.\n\n"
        f"Question: {item['question']}\n{choices}\n\nAnswer:"
    )

def extract_letter(text):
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None

def ask(provider, model, prompt, retries=3):
    client = groq_client if provider == "groq" else router_client
    for attempt in range(retries):
        try:
            kwargs = dict(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            if any(tag in model for tag in REASONING_MODELS):
                kwargs["max_tokens"] = 2048
                kwargs["reasoning_effort"] = "low"
            elif any(tag in model for tag in HYBRID_MODELS):
                kwargs["max_tokens"] = 2048
                kwargs["reasoning_effort"] = "minimal"
            else:
                kwargs["max_tokens"] = 10
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limited on {model}, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    return None

def save_csv():
    # Sort headers so models always appear in the same column order
    sorted_models = sorted(list(all_known_models))
    headers = ["question_id"] + sorted_models + ["correct_label"]
    
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        
        # Write ALL saved_items back to the CSV, keeping old rows perfectly intact
        for item in saved_items:
            q_id = item["question_id"]
            row = {"question_id": q_id, "correct_label": item["answer"]}
            
            for model in sorted_models:
                row[model] = existing_data.get(q_id, {}).get(model, "")
            writer.writerow(row)


In [ ]:

print("\nStarting evaluation loop...")


for idx, item in enumerate(saved_items[:N_QUESTIONS]):
    q_id = item["question_id"]
    prompt = format_question(item)  
    
    for provider, model in MODELS:
        saved_answer = existing_data[q_id].get(model)
        
        if saved_answer:
            continue
            
        print(f"Fetching {q_id} for {model} ({provider})...")
        answer_text = ask(provider, model, prompt)
        predicted = extract_letter(answer_text or "")
        
        # Save to memory and immediately write to CSV
        existing_data[q_id][model] = predicted or "N/A"
        save_csv()
        
        print(f"  -> Predicted: {predicted} | Expected: {item['answer']}| Raw: {answer_text} |") #| Raw: {answer_text} |
        time.sleep(2.5)